Developer acceptance on physical 2×RTX3060. Deliberate test-only CUDA999, bounded short-prefix, and real negative native-plan tests. Requires a C compiler for the small test interposer; ordinary users do not need a compiler. Installs the exact public GitHub wheel. Writes into a new temporary directory; no existing user calculation is resumed or deleted.

In [ ]:
import hashlib,importlib.metadata,json,os,platform,subprocess,sys,time,traceback,urllib.request
from pathlib import Path
import tempfile
RUN_ROOT=Path(tempfile.mkdtemp(prefix='mgbfs-measured-acceptance-'))
ROOT=RUN_ROOT/'output';ROOT.mkdir();WORK=RUN_ROOT/'work';WORK.mkdir()
print('Acceptance output:',ROOT,flush=True)
MANIFEST={'schema': 1, 'wheel': 'multigpubfs-0.2.0.dev29-py3-none-any.whl', 'sha256': 'ab5d6452dd1588bf4c1a003ad911b4cea03d3e6e61e7a6ea3cc18d2d40bf7a51', 'size': 144632, 'source_commit': '9ab794acb1ae78614517ca8a8c06748e9613ed51', 'native_index_sha256': '2d8104d9428e94cf133ad5e8cf93498709077f5c11c46c2e85090151fa0b7eec', 'native_components': 'native-components-20261017-v5', 'status': 'BUILT_NEEDS_INSTALLED_GPU_ACCEPTANCE'}
wheel=WORK/MANIFEST['wheel'];urllib.request.urlretrieve('https://github.com/TryDotAtwo/MultiGPUBFS/releases/download/native-pip-20261017-v29/'+MANIFEST['wheel'],wheel)
assert hashlib.sha256(wheel.read_bytes()).hexdigest()==MANIFEST['sha256']
subprocess.run([sys.executable,'-m','pip','install','--no-deps',str(wheel)],check=True)
assert importlib.metadata.version('multigpubfs')=='0.2.0.dev29'
os.environ['MGBFS_NATIVE_CACHE']=str(WORK/'native-cache')
os.environ.pop('MGBFS_EXECUTABLE',None)
from multigpubfs.native_distribution import native_runtime
native,env=native_runtime()
binary_sha=hashlib.sha256(Path(native).read_bytes()).hexdigest()
assert binary_sha=='10e423aee53f450d5a14f064db0cb1c7784e8080e7b94045adbad378ad304fce'
assert json.loads(subprocess.check_output([native,'key-info'],env=env,text=True))['resident_plan_rejection_v1'] is True
hardware=subprocess.check_output(['nvidia-smi','--query-gpu=name,compute_cap,driver_version','--format=csv,noheader'],text=True).strip().splitlines()
assert len(hardware)==2 and all('3060' in item for item in hardware)
receipt={'schema':1,'status':'RUNNING','package':'0.2.0.dev29','package_source_commit':MANIFEST['source_commit'],'wheel_sha256':MANIFEST['sha256'],'native_binary_sha256':binary_sha,'hardware':hardware,'scope':'Physical 2xRTX3060 measured selection on matrix16/i64 and matrix64/i64 graphs with six generators and 83232 states; original eight-GPU CUDA999 cause remains unknown','checks':[]}
path=ROOT/'measured-tuning-acceptance.json';path.write_text(json.dumps(receipt,indent=2))
child=WORK/'child.py';child.write_text("import contextlib,hashlib,json,os,sys,time,warnings\nfrom collections import Counter\nfrom pathlib import Path\nfrom multigpubfs import GraphDefinition,persistent_graph_workers,run_graph\nfrom multigpubfs.native_distribution import native_runtime\ncase,folder,receipt=sys.argv[1:];family,mode=case.split('-',1);side={'matrix16':4,'matrix64':8}[family];folder=Path(folder);folder.mkdir()\nmod=17\nsmall=[(1,1,0,1),(1,16,0,1),(1,0,1,1),(1,0,16,1)]\ndef mul(a,b):return tuple(sum(a[i*2+k]*b[k*2+j] for k in range(2))%mod for i in range(2) for j in range(2))\nseen={(1,0,0,1)};front=set(seen);first=[]\nwhile front:\n first.append(len(front));children={mul(g,s) for s in front for g in small}-seen;seen.update(children);front=children\nassert len(seen)==4896\nsecond=[1]+[2]*8;oracle=[0]*(len(first)+len(second)-1)\nfor i,a in enumerate(first):\n for j,b in enumerate(second):oracle[i+j]+=a*b\nassert sum(oracle)==83232\nidentity=[int(i==j) for i in range(side) for j in range(side)]\ngenerators=[]\nfor matrix in small:\n full=identity.copy()\n for i in range(2):\n  for j in range(2):full[i*side+j]=matrix[i*2+j]\n generators.append((full,mod))\nfor value in [1,16]:\n full=identity.copy();full[2*side+3]=value;generators.append((full,mod))\n# No supplied orbit bound or forced mode: exercise the real matrix startup\n# selector across admitted HASH/SORTED_RUNS and transport candidates.\ngraph=GraphDefinition.matrix(side,side,generators,identity,name='SL2(17) x C17 exact product')\nnative,env=native_runtime();env=dict(env,MGBFS_TRACE_NATIVE_CALLS='1')\nif mode=='first-candidate-fault':env['MGBFS_TEST_TUNING_FAULT']='probe'\nif mode=='short-prefix':\n import importlib\n launch_module=importlib.import_module('multigpubfs.launch');original_profile_run=launch_module.run_graph\n def bounded_short_probe(graph,target,*args,**kwargs):\n  if Path(target).name=='pilot-3':kwargs['_profile_layers']=10\n  return original_profile_run(graph,target,*args,**kwargs)\n launch_module.run_graph=bounded_short_probe\nnegative_admission_calls=[]\nif mode=='admission-reject':\n import importlib\n tuning_module=importlib.import_module('multigpubfs.autotune')\n original_choose=tuning_module.choose_profile;original_admit=tuning_module._admit\n def measured_cuda_choices(*args,**kwargs):\n  original_available=tuning_module._gemm_hardware_available;tuning_module._gemm_hardware_available=lambda:False\n  try:return original_choose(*args,**kwargs)\n  finally:tuning_module._gemm_hardware_available=original_available\n def native_optional_negative_plan(*args,**kwargs):\n  if args[5].get('MGBFS_GENERIC_GENERATOR')=='gemm' and Path(args[6]).name.startswith('mgbfs-size-profile-'):\n   # Real native admission only: no BFS allocation is requested for this size.\n   negative_admission_calls.append({'generator':'gemm','selected_capacity':args[2],'negative_probe_capacity':1000000000})\n   probe=list(args);probe[2]=1000000000\n   return original_admit(*probe,**kwargs)\n  return original_admit(*args,**kwargs)\n tuning_module.choose_profile=measured_cuda_choices;tuning_module._admit=native_optional_negative_plan\nstarted=time.monotonic()\nwith persistent_graph_workers():\n with warnings.catch_warnings(record=True) as messages:\n  warnings.simplefilter('always')\n  report=run_graph(graph,folder/'first',devices=[0,1],capacity=100000,max_seconds=240,_native_env=env)\n assert report['status']=='COMPLETE' and report['layer_sizes']==oracle,report\n assert report['plan']['elements']==side*side and report['plan']['state_bytes']==8\n profile=report['autotune'];selection=profile.get('global_profile',profile)\n assert profile['measured'] is True and selection['measured'] is True,profile\n assert selection['status']=='MEASURED_EQUAL_PREFIX_GPU_PROFILE',selection\n pilots=selection['pilots'];assert len(pilots)>=2\n if side==8 and mode!='admission-reject':assert any(p.get('candidate')=='generator-gemm' and p.get('generator_backend')=='gemm' for p in pilots),('GEMM_NOT_MEASURED',pilots)\n if mode=='short-prefix':assert any(p['candidate']=='pilot-3' and p['completed_depth']==10 for p in selection['excluded_prefix_candidates'])\n depth=selection['common_depth'];assert depth>=4 and sum(selection['common_layer_sizes'][2:])>=32768\n assert all(p['layer_sizes'][:depth+1]==oracle[:depth+1] for p in pilots)\n assert len(selection['scores_seconds'])==len(pilots)\n assert all(p['rank_worker_pids']==report['rank_worker_pids'] or len(p['rank_worker_pids'])==2 for p in pilots)\n assert 'tuning_failure' not in profile and 'CONSERVATIVE_PROFILE' not in profile['status']\n if mode=='admission-reject':\n  assert len(negative_admission_calls)==1\n  assert len(profile.get('size_generator_rejections',[]))==1\n  assert all(p['generator_backend']=='cuda' for p in profile['size_profiles'])\n  assert any(code in profile['size_generator_rejections'][0]['reason'] for code in ['REQUESTED_CAPACITY_EXCEEDS_SORTED_ADMISSION','REQUESTED_CAPACITY_EXCEEDS_ADMISSION','GEMM_BATCH_EXCEEDS_ADMISSION'])\n failures=selection.get('candidate_failures',[])\n if mode=='first-candidate-fault':\n  assert len(failures)==1 and failures[0]['candidate']=='pilot-0',failures\n  failure=failures[0];evidence=Path(failure['evidence']);assert evidence.is_dir()\n  assert str(evidence) in failure['error'] and failure['probe_retried'] is False\n  primary=json.loads((evidence/'rank-1/phase.json').read_text())['native_error']\n  assert primary['code']==999 and primary['layer']==2\n  summary=json.loads((evidence/'summary.json').read_text())\n  failed_pids=sorted({r['pid'] for c in summary['captures'] for r in c['ranks']})\n  assert set(failed_pids).isdisjoint(report['rank_worker_pids'])\n  assert all(p['candidate']!='pilot-0' for p in pilots)\n  assert any('rejected candidate' in str(m.message) for m in messages)\n else:assert not failures\n again=run_graph(graph,folder/'second',devices=[0,1],capacity=100000,max_seconds=240,_native_env=env)\n assert again['status']=='COMPLETE' and again['layer_sizes']==oracle\n assert again['autotune']['measured'] is True and again['autotune'].get('session_profile_hit') is True,again['autotune']\n assert again['rank_worker_pids']==report['rank_worker_pids']\n public_failures=[{'candidate':f['candidate'],'code':f['native_error']['code'],'rank':f['native_error'].get('rank'),'layer':f['native_error'].get('layer'),'probe_retried':f['probe_retried'],'durable_path_verified':Path(f['evidence']).is_dir()} for f in failures]\n native_plan_refusals=[]\n import multigpubfs.generic_session as session_module\n for worker in session_module._active.workers:\n  log=worker.root/'worker.log'\n  for line in log.read_text().splitlines():\n   if line.startswith('MGBFS_SESSION_PLAN_REJECTED '):native_plan_refusals.append({'rank':worker.rank,'message':line})\n if mode=='admission-reject':\n  assert native_plan_refusals,'real nonfatal native plan rejection marker missing'\n  assert all(worker.process.poll() is None for worker in session_module._active.workers)\n value={'schema':1,'status':'PASS','mode':case,'fixture_elements':side*side,'native_plan_refusals':native_plan_refusals,'size_generator_rejections':profile.get('size_generator_rejections',[]),'negative_admission_calls':negative_admission_calls,'excluded_prefix_candidates':selection.get('excluded_prefix_candidates',[]),'states':sum(oracle),'layer_sizes':oracle,'graph':graph.to_dict(),'graph_digest':graph.digest(),'first_bfs_seconds':report['bfs_seconds'],'autotune_seconds':profile['seconds'],'second_bfs_seconds':again['bfs_seconds'],'second_autotune_seconds':again['autotune']['seconds'],'worker_pids':report['rank_worker_pids'],'second_worker_pids':again['rank_worker_pids'],'measured':True,'selection_measured':profile.get('selection_measured'),'size_profiles_measured':profile.get('size_profiles_measured'),'selected':{k:selection.get(k) for k in ['backend','history_algorithm','owner_lanes','shards','batch','batch_fraction','transport','candidate_order','generator_backend','common_depth','common_layer_sizes','scores_seconds']},'pilots':pilots,'candidate_failures':public_failures,'main_complete':True,'session_profile_reused':True,'wall_seconds':time.monotonic()-started}\n if failures:value['failed_worker_pids']=failed_pids\n Path(receipt).write_text(json.dumps(value,indent=2));print(json.dumps({k:v for k,v in value.items() if k not in ('graph','pilots','layer_sizes','selected')}),flush=True)\n")
inject=WORK/'inject.c';inject.write_text('#define _GNU_SOURCE\n#include <dlfcn.h>\n#include <stdio.h>\n#include <stdlib.h>\n#include <string.h>\nstatic int injected=0;\ntypedef int (*copy_fn)(void*,const void*,size_t,int,void*);\ntypedef int (*sync_fn)(void);\nstatic int eligible(void) {\n const char *mode=getenv("MGBFS_TEST_TUNING_FAULT"),*rank=getenv("RANK"),*path=getenv("MGBFS_JOB_PHASE_PATH");\n if(!mode||!rank||strcmp(rank,"1")||!path)return 0;\n FILE *f=fopen(path,"r");if(!f)return 0;char phase[16384];size_t n=fread(phase,1,sizeof(phase)-1,f);fclose(f);phase[n]=0;\n if(!strstr(phase,"\\"layer\\":2")||!strstr(phase,"\\"native_call_line\\":275"))return 0;\n int pilot=strstr(path,"/pilot-0/")!=NULL;\n const char *leaf=strrchr(path,\'/\');\n if(leaf&&!strncmp(leaf+1,"phase-",6)) {\n  char job[4096];snprintf(job,sizeof(job),"%.*s/job-%s",(int)(leaf-path),path,leaf+7);\n  f=fopen(job,"r");if(f){char text[65536];n=fread(text,1,sizeof(text)-1,f);fclose(f);text[n]=0;pilot=strstr(text,"/pilot-0/")!=NULL;}\n }\n return (!strcmp(mode,"probe")&&pilot)||(!strcmp(mode,"main")&&!pilot);\n}\nint cudaMemcpyAsync(void *dst,const void *src,size_t bytes,int kind,void *stream) {\n if(!injected&&bytes==4&&kind==2&&eligible()){injected=1;return 999;}\n copy_fn real=(copy_fn)dlsym(RTLD_NEXT,"cudaMemcpyAsync");return real(dst,src,bytes,kind,stream);\n}\nint cudaDeviceSynchronize(void) {\n sync_fn real=(sync_fn)dlsym(RTLD_NEXT,"cudaDeviceSynchronize");int result=real();\n return injected&&result==0?901:result;\n}\n')
library=WORK/'inject.so';subprocess.run(['cc','-shared','-fPIC','-O2',str(inject),'-ldl','-o',str(library)],check=True)
try:
 for mode in ['matrix16-healthy','matrix16-first-candidate-fault','matrix64-healthy','matrix64-first-candidate-fault','matrix64-short-prefix','matrix64-admission-reject']:
  mode_env=dict(os.environ,LD_PRELOAD=str(library),MGBFS_PROFILE_CACHE=str(WORK/('profiles-'+mode)))
  output=ROOT/(mode+'.json');started=time.monotonic()
  run=subprocess.run([sys.executable,str(child),mode,str(WORK/mode),str(output)],env=mode_env,capture_output=True,text=True,timeout=360)
  (ROOT/(mode+'.log')).write_text(run.stdout+'\n'+run.stderr)
  print(mode,run.returncode,run.stdout[-2500:],run.stderr[-2500:],flush=True)
  if run.returncode:
   import shutil
   for evidence in (WORK/mode).glob('**/failure-evidence'):shutil.copytree(evidence,ROOT/('failed-'+mode+'-evidence'),dirs_exist_ok=True)
  assert run.returncode==0,(mode,run.returncode)
  data=json.loads(output.read_text());assert data['status']=='PASS';receipt['checks'].append(data)
  path.write_text(json.dumps(receipt,indent=2))
 receipt['status']='PASS'
except BaseException as error:
 receipt.update(status='FAIL',error=str(error));path.write_text(json.dumps(receipt,indent=2));raise
finally:path.write_text(json.dumps(receipt,indent=2))
print('MEASURED_TUNING_ACCEPTANCE',receipt['status'],flush=True)
